# 04. Generation Labeling: Debut Year Coverage

1. Data Loading
2. Merge Debut Years
3. Coverage Check
   - 3.1 Song Share by Artist Type
   - 3.2 Debut Year Coverage
   - 3.3 Coverage by Release Year
   - 3.4 Consistency Check
4. Manual Verification List
5. Apply Manual Updates

## 1. Data Loading

### Objective
Load the song-level dataset (`03_song_level.ipynb`) and the artist debut year table.

In [1]:
from pathlib import Path

import pandas as pd

INTERIM_DIR = Path("../data/interim")
EXTERNAL_DIR = Path("../data/external")

songs = pd.read_pickle(INTERIM_DIR / "songs.pkl")

# Use the manually verified table if it exists (created in section 5)
verified_path = EXTERNAL_DIR / "artist_debut_years_verified.csv"
debut_path = verified_path if verified_path.exists() else EXTERNAL_DIR / "artist_debut_years.csv"
debut = pd.read_csv(debut_path, dtype={"debut_year": "Int64"})

print(f"Songs: {len(songs):,}")
print(f"Debut table: {debut_path.name} ({len(debut):,} artists)")

Songs: 7,988
Debut table: artist_debut_years_verified.csv (1,539 artists)


## 2. Merge Debut Years

### Objective
Attach artist type, debut year, and confidence to each song.

### Procedure
- Merge on the exact `artist` string.
- Report artists in the song data that are missing from the debut table.

In [2]:
songs["artist"] = songs["artist"].str.strip()
debut["artist"] = debut["artist"].str.strip()

df = songs.merge(
    debut[["artist", "artist_type", "debut_year", "confidence", "parent_group", "web_verified", "note"]],
    on="artist", how="left", indicator=True,
)

unmatched = df[df["_merge"] == "left_only"]
print(f"Songs without a match in the debut table: {len(unmatched):,}")
if len(unmatched):
    display(unmatched.groupby("artist").size().sort_values(ascending=False).to_frame("songs"))

df = df.drop(columns="_merge")
df["artist_type"] = df["artist_type"].fillna("unmatched")

Songs without a match in the debut table: 0


## 3. Coverage Check

### 3.1 Song Share by Artist Type

#### Objective
Measure how many songs belong to Korean artists eligible for generation analysis.

#### Procedure
- Count songs by artist type. All shares are song-based, not artist-based.
- Group types:
  - **Eligible:** solo, group, unit
  - **Korean, not eligible:** collab, project, actor, other
  - **Foreign**
  - **Unknown:** unidentified or ambiguous artists
- The Korean share is reported as a range: unknown artists excluded (lower bound) or all counted as Korean (upper bound).

In [3]:
TYPE_GROUP = {
    "solo": "eligible", "group": "eligible", "unit": "eligible",
    "collab": "korean_not_eligible", "project": "korean_not_eligible",
    "actor": "korean_not_eligible", "other": "korean_not_eligible",
    "foreign": "foreign",
    "unknown": "unknown", "unmatched": "unknown",
}
df["type_group"] = df["artist_type"].map(TYPE_GROUP)

def share_table(data, col):
    t = data[col].value_counts().to_frame("songs")
    t["share (%)"] = (t["songs"] / t["songs"].sum() * 100).round(2)
    return t

display(share_table(df, "artist_type"))
display(share_table(df, "type_group"))

n = len(df)
korean_min = df["type_group"].isin(["eligible", "korean_not_eligible"]).sum()
korean_max = korean_min + (df["type_group"] == "unknown").sum()
print(f"Korean artist songs: {korean_min / n:.2%} – {korean_max / n:.2%}")
print(f"Eligible for generation analysis: {(df['type_group'] == 'eligible').mean():.2%}")

,songs,share (%)
artist_type,,
solo,3830,47.95
group,2948,36.91
unknown,635,7.95
foreign,151,1.89
actor,103,1.29
project,101,1.26
unit,99,1.24
other,94,1.18
collab,27,0.34


,songs,share (%)
type_group,,
eligible,6877,86.09
unknown,635,7.95
korean_not_eligible,325,4.07
foreign,151,1.89


Korean artist songs: 90.16% – 98.11%
Eligible for generation analysis: 86.09%


### 3.2 Debut Year Coverage

#### Objective
Measure the share of eligible songs with a debut year, by confidence level.

#### Procedure
- Base: songs by eligible artists (solo, group, unit).
- Coverage = songs with a debut year / eligible songs.
- Target: 90% or higher.

In [4]:
def debut_coverage(data):
    el = data[data["type_group"] == "eligible"]
    has_year = el["debut_year"].notna()
    conf = el["confidence"].fillna("U").where(has_year, "missing")
    t = conf.value_counts().reindex(["H", "M", "L", "missing"], fill_value=0).to_frame("songs")
    t["share (%)"] = (t["songs"] / len(el) * 100).round(2)
    return t, has_year.mean()

cov_table, coverage = debut_coverage(df)
display(cov_table)
print(f"Debut year coverage (eligible songs): {coverage:.2%}")
print(f"High-confidence or web-verified coverage: "
      f"{df.loc[df.type_group == 'eligible', 'confidence'].eq('H').mean():.2%}")

,songs,share (%)
confidence,,
H,5665,82.38
M,525,7.63
L,275,4.00
missing,412,5.99


Debut year coverage (eligible songs): 94.01%
High-confidence or web-verified coverage: 82.38%


### 3.3 Coverage by Release Year

#### Objective
Check whether missing debut years are concentrated in specific periods, which would bias generation comparisons.

#### Procedure
- For eligible songs released in 2000–2023, compute debut year coverage per release year.

In [5]:
el = df[(df["type_group"] == "eligible") & df["release_year"].between(2000, 2023)]

by_year = el.groupby("release_year").agg(
    eligible_songs=("song_key", "size"),
    with_debut_year=("debut_year", lambda s: s.notna().sum()),
)
by_year["coverage (%)"] = (by_year["with_debut_year"] / by_year["eligible_songs"] * 100).round(1)

display(by_year)
print(f"Lowest coverage: {by_year['coverage (%)'].idxmin()} ({by_year['coverage (%)'].min()}%)")

,eligible_songs,with_debut_year,coverage (%)
release_year,,,
2000,168,153,91.1
2001,169,154,91.1
2002,161,153,95.0
2003,175,159,90.9
2004,209,202,96.7
2005,345,328,95.1
2006,333,295,88.6
2007,318,289,90.9
2008,347,329,94.8


Lowest coverage: 2020 (88.2%)


### 3.4 Consistency Check

#### Objective
Find debut years that cannot be correct.

#### Procedure
- A debut year later than the artist's first chart appearance is an error.
- These artists are added to the verification list with the highest priority.

In [6]:
artist_first_chart = df.groupby("artist")["first_chart_year"].min()

check = (
    debut.set_index("artist")[["artist_type", "debut_year", "confidence"]]
    .join(artist_first_chart.rename("first_chart_year"), how="inner")
)
inconsistent = check[check["debut_year"] > check["first_chart_year"]]

print(f"Artists with debut year after first chart appearance: {len(inconsistent)}")
display(inconsistent)

Artists with debut year after first chart appearance: 16


,artist_type,debut_year,confidence,first_chart_year
artist,,,,
AKMU (악뮤),group,2014,H,2012
B.I,solo,2015,H,2014
BOBBY,solo,2015,H,2014
KATIE,solo,2018,H,2015
강승윤,solo,2011,H,2010
권진아,solo,2016,H,2014
로이킴,solo,2013,H,2012
박보람,solo,2014,H,2010
방예담,solo,2020,H,2013


## 4. Manual Verification List

### Objective
List the artists whose debut years should be checked manually, ordered by impact.

### Procedure
- Include artists that are:
  - eligible with a missing debut year or confidence M/L,
  - unknown or unmatched (they may be eligible Korean artists),
  - inconsistent in 3.4.
- Sort by number of songs; `cum_share` shows the cumulative share of all songs covered.
- Sample song titles and chart years are included to identify ambiguous names.

### Output
- `data/interim/debut_verification_list.csv`

### How to fill in
- `checked_type`: solo / group / unit / collab / project / actor / other / foreign
- `checked_debut_year`: year of the first official debut (including as a group member)
- `source`: where the value was confirmed (e.g. Melon artist page, Wikipedia)
- Leave a row empty to keep the current value.
- Values already filled in are kept when this cell is re-run.

In [7]:
artist_stats = df.groupby("artist").agg(
    n_songs=("song_key", "size"),
    first_chart_year=("first_chart_year", "min"),
    last_chart_year=("last_chart_year", "max"),
    sample_songs=("song_name", lambda s: " / ".join(s.head(3))),
)

cand = (
    debut.set_index("artist")[["artist_type", "debut_year", "confidence", "web_verified", "note"]]
    .join(artist_stats, how="inner")
)
cand = pd.concat([
    cand,
    artist_stats.loc[artist_stats.index.difference(debut["artist"])].assign(artist_type="unmatched"),
])

is_eligible = cand["artist_type"].isin(["solo", "group", "unit"])
needs_check = (
    (is_eligible & (cand["debut_year"].isna() | cand["confidence"].isin(["M", "L"])))
    | cand["artist_type"].isin(["unknown", "unmatched"])
    | cand.index.isin(inconsistent.index)
)

verify = cand[needs_check].copy()
verify["inconsistent"] = verify.index.isin(inconsistent.index)
verify = verify.sort_values(["inconsistent", "n_songs"], ascending=[False, False])
verify["cum_share"] = (verify["n_songs"].cumsum() / len(df)).round(4)
verify["checked_type"] = ""
verify["checked_debut_year"] = ""
verify["source"] = ""

verify = verify.reset_index().rename(columns={"index": "artist"})

# Keep values already filled in by hand when the notebook is re-run
list_path = INTERIM_DIR / "debut_verification_list.csv"
if list_path.exists():
    prev = pd.read_csv(list_path, dtype=str).fillna("")
    prev = prev[(prev["checked_type"] != "") | (prev["checked_debut_year"] != "")]
    prev = prev.set_index("artist")[["checked_type", "checked_debut_year", "source"]]
    verify = verify.set_index("artist")
    verify.update(prev)
    missing_rows = prev.index.difference(verify.index)
    verify = pd.concat([verify, prev.loc[missing_rows]]).reset_index().rename(columns={"index": "artist"})
    print(f"Carried over manual entries: {len(prev)}")

verify["n_songs"] = verify["n_songs"].astype("Int64")
verify.to_csv(list_path, index=False, encoding="utf-8-sig")

print(f"Artists to check: {len(verify):,} ({verify['n_songs'].sum():,} songs, "
      f"{verify['n_songs'].sum() / len(df):.2%} of all songs)")
print("\nArtists needed to cover a given share of these songs:")
for target in [0.5, 0.8, 0.9]:
    k = (verify["n_songs"].cumsum() / verify["n_songs"].sum() < target).sum() + 1
    print(f"  {target:.0%}: top {min(k, len(verify))} artists")

display(verify.head(30))

Carried over manual entries: 50
Artists to check: 751 (2,059 songs, 25.78% of all songs)

Artists needed to cover a given share of these songs:
  50%: top 117 artists
  80%: top 329 artists
  90%: top 512 artists


,artist,artist_type,debut_year,confidence,web_verified,note,n_songs,first_chart_year,last_chart_year,sample_songs,inconsistent,cum_share,checked_type,checked_debut_year,source
0,AKMU (악뮤),group,2014,H,Y,NaN,50,2012.0,2023.0,다리꼬지마 / 매력있어 / 라면인건가,True,0.0063,group,2014,Naver
1,케이윌,solo,2007,H,Y,NaN,48,2005.0,2022.0,꿈 (Dream) / 왼쪽가슴 / 하리오 (Sweet Mix),True,0.0123,solo,2007,Naver
2,버스커 버스커,group,2012,H,Y,NaN,30,2011.0,2023.0,동경소녀 / 정류장 / Livin'La Vida Loca (Ricky Martin),True,0.0160,group,2012,Naver
3,로이킴,solo,2013,H,Y,NaN,19,2012.0,2023.0,먼지가 되어 / 휘파람 / 청개구리,True,0.0184,solo,2013,Naver
4,정승환,solo,2016,H,Y,NaN,11,2014.0,2020.0,사랑에 빠지고 싶다 / 슬픔 속에 그댈 지워야만 해 / 너를 사랑한 시간,True,0.0198,solo,2016,Google
5,유승우,solo,2013,H,Y,NaN,9,2012.0,2016.0,My Son / 헬로 / 너와 나,True,0.0209,solo,2013,Naver
6,존박,solo,2012,H,Y,NaN,8,2010.0,2022.0,내가 다 줄게요 / 빗속에서 / I'm Your Man,True,0.0219,solo,2012,Naver
7,권진아,solo,2016,H,Y,NaN,7,2014.0,2020.0,씨스루 / Love / 끝,True,0.0228,solo,2016,Naver
8,박보람,solo,2014,H,Y,NaN,7,2010.0,2018.0,세월이 가면 / 예뻐졌다 (Feat. 지코 Of Block B) / 연예할래,True,0.0237,solo,2014,Naver
9,홍대광,solo,2013,H,Y,NaN,7,2012.0,2015.0,가족 / 이미 넌 고마운 사람 / 굿바이 (With 소유 Of 씨스타),True,0.0245,solo,2013,Naver


## 5. Apply Manual Updates

### Objective
Apply the manually checked values and save a verified debut table.

### Procedure
- Read `debut_verification_list.csv` after filling in `checked_type`, `checked_debut_year`, and `source`.
- Overwrite the type and debut year for filled rows; set confidence to H and record the source.
- Save as `data/external/artist_debut_years_verified.csv`.
- Re-run the notebook from the top: section 1 loads the verified table automatically, and section 3 shows the updated coverage.

In [8]:
checked = pd.read_csv(INTERIM_DIR / "debut_verification_list.csv", dtype=str).fillna("")
checked = checked[(checked["checked_type"] != "") | (checked["checked_debut_year"] != "")]
print(f"Rows with manual updates: {len(checked)}")

if len(checked):
    updated = debut.set_index("artist").copy()
    updated["debut_year"] = updated["debut_year"].astype("Int64")
    if "source" not in updated.columns:
        updated["source"] = ""

    for _, r in checked.iterrows():
        a = r["artist"]
        if a not in updated.index:
            updated.loc[a] = pd.NA
        if r["checked_type"]:
            updated.loc[a, "artist_type"] = r["checked_type"].strip()
        if r["checked_debut_year"]:
            updated.loc[a, "debut_year"] = int(float(r["checked_debut_year"]))
            updated.loc[a, "confidence"] = "H"
        updated.loc[a, "source"] = r["source"]
        updated.loc[a, "web_verified"] = "Y"

    updated.reset_index().to_csv(verified_path, index=False, encoding="utf-8-sig")
    print(f"Saved {verified_path}")

Rows with manual updates: 50
Saved ../data/external/artist_debut_years_verified.csv


## 6. Generation Assignment

### Objective
Assign K-pop generations from artist debut years under the baseline and alternative boundaries.

### Procedure
- Only eligible artists (solo, group, unit) receive a generation.
- Boundaries (debut year):

| Scheme | 1st | 2nd | 3rd | 4th |
|---|---|---|---|---|
| baseline | 1996–2002 | 2003–2011 | 2012–2017 | 2018–2022 |
| alt_A | 1996–2003 | 2004–2011 | 2012–2017 | 2018–2022 |
| alt_B | 1996–2002 | 2003–2012 | 2013–2017 | 2018–2022 |
| alt_C | 1996–2002 | 2003–2011 | 2012–2018 | 2019–2022 |

- Debut before 1996 → `pre-1st`; debut in 2023 → `5th`. Both are labeled but excluded from the RQ1 comparison by default (`in_rq1`).
- `high_conf`: debut year with confidence H, used for the sensitivity check.
- `in_rq2`: release year 2000–2023, used for the temporal analysis.

In [9]:
SCHEMES = {
    "baseline": [1996, 2003, 2012, 2018, 2023],
    "alt_A":    [1996, 2004, 2012, 2018, 2023],
    "alt_B":    [1996, 2003, 2013, 2018, 2023],
    "alt_C":    [1996, 2003, 2012, 2019, 2023],
}
LABELS = ["1st", "2nd", "3rd", "4th"]
GEN_ORDER = ["pre-1st"] + LABELS + ["5th"]

def assign_generation(year, bounds):
    if pd.isna(year):
        return pd.NA
    if year < bounds[0]:
        return "pre-1st"
    if year >= bounds[-1]:
        return "5th"
    for i in range(4):
        if bounds[i] <= year < bounds[i + 1]:
            return LABELS[i]

eligible = df["type_group"] == "eligible"

for name, bounds in SCHEMES.items():
    df[f"gen_{name}"] = (
        df["debut_year"].where(eligible)
        .apply(assign_generation, bounds=bounds)
        .astype(pd.CategoricalDtype(GEN_ORDER, ordered=True))
    )

df["high_conf"] = eligible & df["confidence"].eq("H") & df["debut_year"].notna()
df["in_rq1"] = df["gen_baseline"].isin(LABELS)
df["in_rq2"] = df["release_year"].between(2000, 2023)

gen_counts = pd.DataFrame({
    name: df[f"gen_{name}"].value_counts().reindex(GEN_ORDER) for name in SCHEMES
})
gen_counts.loc["no generation"] = len(df) - gen_counts.sum()
display(gen_counts)

print(f"RQ1 songs (baseline, 1st–4th): {df['in_rq1'].sum():,}")
print(f"  of which high-confidence: {(df['in_rq1'] & df['high_conf']).sum():,}")
print(f"RQ2 songs (release year 2000–2023): {df['in_rq2'].sum():,}")

,baseline,alt_A,alt_B,alt_C
pre-1st,535,535,535,535
1st,1818,2184,1818,1818
2nd,2736,2370,3154,2736
3rd,1226,1226,808,1274
4th,148,148,148,100
5th,2,2,2,2
no generation,1523,1523,1523,1523


RQ1 songs (baseline, 1st–4th): 5,928
  of which high-confidence: 5,237
RQ2 songs (release year 2000–2023): 7,875


### 6.1 Generation by Release Period

#### Objective
Check how generations are distributed over release years, since generation and release time overlap.

In [10]:
period = pd.cut(df["release_year"], bins=[1999, 2004, 2009, 2014, 2019, 2023],
                labels=["2000–04", "2005–09", "2010–14", "2015–19", "2020–23"])
display(pd.crosstab(df.loc[df["in_rq1"], "gen_baseline"].cat.remove_unused_categories(),
                    period[df["in_rq1"]], margins=True))

release_year,2000–04,2005–09,2010–14,2015–19,2020–23,All
gen_baseline,,,,,,
1st,554,633,420,127,26,1760
2nd,64,793,1279,454,146,2736
3rd,0,0,245,716,265,1226
4th,0,0,1,31,116,148
All,618,1426,1945,1328,553,5870


## 7. Export

### Output
- `data/interim/songs_labeled.pkl`: song-level dataset with artist type, debut year, generations, and analysis flags

In [11]:
label_columns = [
    "artist_type", "type_group", "debut_year", "confidence", "parent_group",
    *[f"gen_{name}" for name in SCHEMES], "high_conf", "in_rq1", "in_rq2",
]
out = songs.join(df.set_index("song_key")[label_columns], on="song_key")
out.to_pickle(INTERIM_DIR / "songs_labeled.pkl")

print(f"Saved {len(out):,} songs to {INTERIM_DIR / 'songs_labeled.pkl'}")

Saved 7,988 songs to ../data/interim/songs_labeled.pkl
